# Benders Code

In [57]:
import gurobipy as gp
from gurobipy import GRB

def solve_benders_labour_budgets(): # Renamed function
    # 1) DATA - Adjusted further for complexity and added constraints
    machines        = ['AM_hWAAM', 'AM_Binder', 'AM_PBF', 'TM_Casting']

    # --- scenarios ---
    scenarios       = ['s1', 's2', 's3', 's4', 's5', 's6', 's7']
    demand_year2    = {s: d for s, d in zip(scenarios, [400, 600, 900, 1200, 1600, 2000, 2500])}
    prob_scenario   = {s: p for s, p in zip(scenarios, [0.05, 0.10, 0.15, 0.25, 0.20, 0.10, 0.15])}

    # ---  Year 1 demand ---
    demand_year1    = 1000

    # ---  Year 1 Budget ---
    budget_year1    = 15000 # Capped at 15k for Year 1

    # --- Year 2 Budget ---
    max_budget_year2 = 15000 # Capped at 15k for Year 2 extra purchases


    batch_size      = {'AM_hWAAM':6, 'AM_Binder':6, 'AM_PBF':5, 'TM_Casting':1}
    time_per_batch  = {'AM_hWAAM':9.30, 'AM_Binder':8.74, 'AM_PBF':22.86, 'TM_Casting':6.0}

    purchase_cost   = {'AM_hWAAM':1800, 'AM_Binder':3800, 'AM_PBF':7000, 'TM_Casting':5500}

    hours_per_month = {'AM_hWAAM':200,  'AM_Binder':280,  'AM_PBF':280,   'TM_Casting':200}
    hours_per_year  = {m: hours_per_month[m]*12 for m in machines}

    material_cost   = 50.0
    operating_cost  = {'AM_hWAAM':200, 'AM_Binder':150, 'AM_PBF':110, 'TM_Casting':40}

    # --- Labour Data ---
    labour_per_batch = {'AM_hWAAM':3.0, 'AM_Binder':2.5,
                       'AM_PBF':4.0, 'TM_Casting':1.0}  # person-hours

    max_labour_year  = 7000 # Year 1 labor limit
    max_labour_year2 = 8000 # Year 2 labor limit (per scenario)

    am_machines = ['AM_hWAAM', 'AM_Binder', 'AM_PBF']
    eos_mult = {}
    
    # --- EOS multipliers ---
    eos_mult_am_values = [0.80, 0.88, 0.95, 1.00, 1.05, 1.12, 1.20]
    eos_mult_tm_values = [1.40, 1.25, 1.10, 1.00, 0.90, 0.80, 0.70]
    eos_mult_map = {s: {'AM': am_v, 'TM': tm_v} for s, am_v, tm_v in zip(scenarios, eos_mult_am_values, eos_mult_tm_values)}

    for m in machines:
         machine_type = 'AM' if m in am_machines else 'TM'
         for s in scenarios:
             eos_mult[(m,s)] = eos_mult_map[s][machine_type]

    # Year‑2 batch cost including economies of scale & global price swings
    var_cost_year2 = {}
    for m in machines:
        base = batch_size[m]*material_cost + time_per_batch[m]*operating_cost[m]
        for s in scenarios:
            var_cost_year2[(m,s)] = base  * eos_mult[(m,s)]

    # 2) MASTER
    master = gp.Model("Master_LabourBudgets") # Renamed model
    master.Params.OutputFlag = 0

    # first‑stage vars
    machines_bought = master.addVars(machines, vtype=GRB.INTEGER, lb=0, name="n")
    batches_year1   = master.addVars(machines, vtype=GRB.CONTINUOUS, lb=0, name="z1")
    theta = master.addVar(lb=-GRB.INFINITY, name="theta")
    master.addConstr(theta >= 0.0, name="theta_non_negative")


    # first‑stage constraints
    master.addConstr(
        gp.quicksum(batch_size[m] * batches_year1[m] for m in machines)
        >= demand_year1, name="demand1"
    )
    for m in machines:
        master.addConstr(
            time_per_batch[m] * batches_year1[m]
            <= hours_per_year[m] * machines_bought[m],
            name=f"cap1_{m}"
        )
    master.addConstr(
        gp.quicksum(purchase_cost[m] * machines_bought[m] for m in machines)
        <= budget_year1, name="budget1"
    )
    # --- Added Year 1 Labour constraint ---
    master.addConstr(
        gp.quicksum(labour_per_batch[m] * batches_year1[m] for m in machines)
        <= max_labour_year, name="labour1")


    # objective = first‑stage fixed + var + theta
    fixed1 = gp.quicksum(purchase_cost[m]*machines_bought[m] for m in machines)
    var1   = gp.quicksum(
        (batch_size[m]*material_cost + time_per_batch[m]*operating_cost[m])
        * batches_year1[m] for m in machines
    )
    master.setObjective(fixed1 + var1 + theta, GRB.MINIMIZE)
    master.update()

    # 3) BENDERS LOOP
    LB, UB = -1e10, 1e10
    tol = 1e-4
    best_n = None

    print("Starting Benders Decomposition (Labour and Budgets)...")

    for it in range(1, 101):
        # Revert Gurobi parameter changes, rely on new constraints to add complexity
        #master.Params.Presolve = -1
        #master.Params.Heuristics = 0
        master.optimize()

        if master.Status == GRB.OPTIMAL:
            n_val = {m: machines_bought[m].X for m in machines}
            LB = max(LB, master.ObjVal)
        elif master.Status == GRB.INF_OR_UNBD:
             print("\nMaster problem is infeasible or unbounded. Benders terminated.")
             break
        else:
             print(f"\nMaster problem did not solve to optimality. Status: {master.Status}. Benders terminated.")
             break

        is_integral_sol = all(abs(machines_bought[m].X - round(machines_bought[m].X)) < 1e-5 for m in machines)


        exp2 = 0.0
        alpha_duals = {} # Duals for demand2_sub (>=)
        beta_duals = {}  # Duals for cap2_m_sub (<=)
        gamma_duals = {} # Duals for labour2_sub (<=)
        delta_duals = {} # Duals for budget2_sub (<=)

        sub_issue = False

        for s in scenarios:
            sub = gp.Model(f"Sub_{s}_LabourBudgets") # Renamed model
            sub.Params.OutputFlag = 0

            # recourse vars (y2 machines, z2 batches)
            extra = sub.addVars(machines, vtype=GRB.CONTINUOUS, lb=0, name="y2")
            z2    = sub.addVars(machines, vtype=GRB.CONTINUOUS, lb=0, name="z2")

            # Recourse constraints
            sub.addConstr(
                gp.quicksum(batch_size[m] * z2[m] for m in machines)
                >= demand_year2[s],
                name="demand2_sub"
            )

            #Capacity 2 (using total machines: Year 1 bought + Year 2 extra)
            for m in machines:
                sub.addConstr(
                    time_per_batch[m] * z2[m]
                    <= hours_per_year[m] * (n_val[m] + extra[m]),
                    name=f"cap2_{m}_sub"
                )
            # --- Added Year 2 Labour constraint ---
            sub.addConstr(
                gp.quicksum(labour_per_batch[m] * z2[m] for m in machines)
                <= max_labour_year2, name="labour2_sub")

            # --- Added Year 2 Budget constraint for extra purchases ---
            sub.addConstr(
                 gp.quicksum(purchase_cost[m] * extra[m] for m in machines)
                 <= max_budget_year2, name="budget2_sub")


            # Recourse objective (Cost in scenario s)
            sub.setObjective(
                gp.quicksum(purchase_cost[m] * extra[m] + var_cost_year2[(m, s)] * z2[m]
                            for m in machines),
                GRB.MINIMIZE
            )
            sub.optimize()

            if sub.Status == GRB.OPTIMAL:
                exp2 += prob_scenario[s] * sub.ObjVal
                alpha_duals[s] = sub.getConstrByName("demand2_sub").Pi
                beta_duals[s] = {m: sub.getConstrByName(f"cap2_{m}_sub").Pi for m in machines}
                gamma_duals[s] = sub.getConstrByName("labour2_sub").Pi # Added dual retrieval
                delta_duals[s] = sub.getConstrByName("budget2_sub").Pi # Added dual retrieval


            elif sub.Status == GRB.INFEASIBLE:
                 print(f"Subproblem for scenario {s} is infeasible with n = {n_val}. Cannot meet year 2 demand.")
                 sub_issue = True
                 # Infeasibility cut logic would be needed here if infeasibility is possible
                 break
            elif sub.Status == GRB.UNBOUNDED:
                 print(f"Subproblem for scenario {s} is unbounded with n = {n_val}.")
                 sub_issue = True
                 # Unboundedness cut logic would be needed here if unboundedness is possible
                 break
            else:
                 print(f"Subproblem for scenario {s} did not solve to optimality. Status: {sub.Status}")
                 sub_issue = True
                 break

        if sub_issue:
            print("Encountered subproblem issue. Benders terminated.")
            break

        if is_integral_sol:
            UB_cand = fixed1.getValue() + var1.getValue() + exp2
            if UB_cand < UB:
                UB = UB_cand
                best_n = n_val.copy()

        # Add Optimality Cut 
        # theta >= Sum_s prob_s * [alpha_s*D2[s] + sum_m beta_s_m*HPY[m]*n[m] + gamma_s*max_labour2 + delta_s*max_budget2]
        cut_expr = gp.LinExpr()
        for s in scenarios:
            # Constant terms from duals of constraints with constant RHS
            cut_expr += prob_scenario[s] * alpha_duals[s] * demand_year2[s]
            cut_expr += prob_scenario[s] * gamma_duals[s] * max_labour_year2 # Added labour dual contribution
            cut_expr += prob_scenario[s] * delta_duals[s] * max_budget_year2 # Added budget dual contribution

            # Term from capacity dual linked to master variable n (using +=, duals beta_s_m are <= 0)
            for m in machines:
                 cut_expr += prob_scenario[s] * beta_duals[s][m] * hours_per_year[m] * machines_bought[m]

        print(f"Iter {it}: Adding cut: theta >= {cut_expr}") # Can be long

        master.addConstr(theta >= cut_expr, name=f"bcut_{it}")
        master.update()

        gap = abs(UB - LB) / max(1e-6, abs(UB))
        print(f"Iter {it}: LB={LB:.2f}, UB={UB:.2f}, gap={100*gap:.2f}%")

        if gap < tol:
            print(f"Convergence reached with gap {100*gap:.2f}% < tolerance {100*tol:.2f}%")
            # Re-solve master one last time to confirm optimal integer solution
            #master.Params.Presolve = -1 # Reset parameters for final solve
            #master.Params.Heuristics = 0
            master.optimize()
            if master.Status == GRB.OPTIMAL:
                 final_n_val_check = {m: machines_bought[m].X for m in machines}
                 if all(abs(final_n_val_check[m] - round(final_n_val_check[m])) < 1e-5 for m in machines):
                     print("Verified optimal master solution is integral.")
                     LB = master.ObjVal # Final LB from optimal master
                     UB = min(UB, LB) # Ensure UB is at least LB
                     break # Exit loop
                 else:
                     print("Warning: Optimal master solution is fractional despite small gap. Continuing iterations.")
            else:
                 print("Warning: Could not re-solve master to verify integrality. Continuing iterations.")


    # 4) REPORT
    print("\nFinished Benders decomposition (Labour and Budgets).")
    print("Iterations:", it)
    print("Final LB:", LB)
    print("Final UB:", UB)

    #master.Params.Presolve = -1 # Reset parameters for final reporting solve
    #master.Params.Heuristics = 0
    #master.optimize() # Final solve for reporting

    if master.Status == GRB.OPTIMAL:
        final_n_val = {m: machines_bought[m].X for m in machines}
        print("\nOptimal expected total cost:", master.ObjVal)
        print("\nOptimal Year‑1 machine purchases:")
        for m in machines:
            print(f"   {m}     : {round(final_n_val[m])}") # Round for display as they should be integer

        print("\nExtra purchases if scenario occurs (y2) and Year 2 batches (z2):")

        for s in scenarios:
            sub = gp.Model(f"Optimal_Sub_{s}_LabourBudgets") # Renamed
            sub.Params.OutputFlag = 0

            extra = sub.addVars(machines, vtype=GRB.CONTINUOUS, lb=0, name="y2")
            z2    = sub.addVars(machines, vtype=GRB.CONTINUOUS, lb=0, name="z2")

            sub.addConstr(
                gp.quicksum(batch_size[m] * z2[m] for m in machines)
                >= demand_year2[s],
                name="demand2_sub"
            )
            for m in machines:
                sub.addConstr(
                    time_per_batch[m] * z2[m]
                    <= hours_per_year[m] * (final_n_val[m] + extra[m]), # Use final_n_val here
                    name=f"cap2_{m}_sub"
                )
            sub.addConstr(
                gp.quicksum(labour_per_batch[m] * z2[m] for m in machines)
                <= max_labour_year2, name="labour2_sub") # Added

            sub.addConstr(
                 gp.quicksum(purchase_cost[m] * extra[m] for m in machines)
                 <= max_budget_year2, name="budget2_sub") # Added


            sub.setObjective(
                gp.quicksum(purchase_cost[m] * extra[m] + var_cost_year2[(m, s)] * z2[m]
                            for m in machines),
                GRB.MINIMIZE
            )
            sub.optimize()

            if sub.Status == GRB.OPTIMAL:
                optimal_year2_purchases = {m: extra[m].X for m in machines}
                # optimal_year2_batches = {m: z2[m].X for m in machines} # Optional: store batches

                y2_output = ", ".join([f"{m}={optimal_year2_purchases[m]:.0f}" for m in machines])
                print(f"    {s}: {y2_output}")

            elif sub.Status == GRB.INFEASIBLE:
                 print(f"Error: Optimal master solution leads to infeasible subproblem for scenario {s}.")
            elif sub.Status == GRB.UNBOUNDED:
                 print(f"Error: Optimal master solution leads to unbounded subproblem for scenario {s}.")
            else:
                 print(f"Error: Optimal subproblem for scenario {s} did not solve to optimality. Status: {sub.Status}")

    else:
        print("\nCould not obtain final optimal solution from master.")
        if best_n is not None:
             print("\nBest integral solution found during Benders loop:")
             print("Optimal expected total cost (UB from loop):", UB)
             print("\nOptimal Year‑1 machine purchases:")
             for m in machines:
                 print(f"    {m}     : {round(best_n[m])}")


if __name__ == "__main__":
    solve_benders_labour_budgets()


Starting Benders Decomposition (Labour and Budgets)...
Iter 1: Adding cut: theta >= 379825.28863636364 + -90.00000000000001 n[AM_hWAAM] + 0.0 n[AM_Binder] + -350.00000000000006 n[AM_PBF] + -275.0 n[TM_Casting] + -180.00000000000003 n[AM_hWAAM] + 0.0 n[AM_Binder] + -700.0000000000001 n[AM_PBF] + -550.0 n[TM_Casting] + -270.0 n[AM_hWAAM] + 0.0 n[AM_Binder] + -1050.0 n[AM_PBF] + -824.9999999999999 n[TM_Casting] + -450.0 n[AM_hWAAM] + 0.0 n[AM_Binder] + -1750.0000000000002 n[AM_PBF] + -1375.0 n[TM_Casting] + -547.8545454545454 n[AM_hWAAM] + 0.0 n[AM_Binder] + -2130.5454545454545 n[AM_PBF] + -1673.9999999999998 n[TM_Casting] + -899.607272727273 n[AM_hWAAM] + 0.0 n[AM_Binder] + -3498.472727272729 n[AM_PBF] + -2748.800000000001 n[TM_Casting] + -2340.6545454545444 n[AM_hWAAM] + 0.0 n[AM_Binder] + -9102.54545454545 n[AM_PBF] + -7151.999999999998 n[TM_Casting]
Iter 1: LB=272300.00, UB=652125.29, gap=58.24%
Iter 2: Adding cut: theta >= 379094.74318181816 + -90.00000000000001 n[AM_hWAAM] + 0.0 n[A